# AI Handwriting Generator — Training Notebook
**Step 0:** Runtime → Change runtime type → **T4 GPU** before running.

This notebook trains the `HandwritingCNN` regression model (Bézier curve predictor).
After training, download `checkpoints/generator.pt` and place it in your local `checkpoints/` folder.

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install opencv-contrib-python reportlab scikit-image -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone / pull repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

os.chdir(REPO_PATH)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 5: Copy Writers_pngs from Drive
# Upload your Writers_pngs folder to: My Drive/Writers_pngs/
import shutil, os

src  = '/content/drive/MyDrive/Writers_pngs'
dest = f'{REPO_PATH}/Data/Writers_pngs'

if os.path.exists(dest) and len(os.listdir(dest)) > 2:
    print('Writers_pngs already present — skipping copy')
elif os.path.exists(src):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print('Writers_pngs copied successfully')
else:
    print('ERROR: Writers_pngs not found at', src)
    raise FileNotFoundError(src)

skip = {'Writers_Zip', 'output_preview'}
total = 0
for d in sorted(os.listdir(dest)):
    if d in skip: continue
    p = os.path.join(dest, d)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f'  {d}: {n} samples')
        total += n
print(f'Total: {total} samples')

In [ ]:
# Cell 6: Train the model
import sys
sys.path.insert(0, f'{REPO_PATH}/src')

# Override epochs for Colab (more epochs = better quality)
import src.train as tr
tr.NUM_EPOCHS = 80
tr.DATA_ROOT  = f'{REPO_PATH}/Data/Writers_pngs'
tr.CKPT_DIR   = f'{REPO_PATH}/checkpoints'

tr.main()

In [ ]:
# Cell 7: Save checkpoint to Drive + download
import shutil, os
from google.colab import files

ckpt_src  = f'{REPO_PATH}/checkpoints/generator.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'generator.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('Download started — put generator.pt into your local checkpoints/ folder')

In [ ]:
# Cell 8: Quick generation demo in Colab
import sys, os
sys.path.insert(0, f'{REPO_PATH}/src')

from generate import load_model, generate_word
from PIL import Image
import matplotlib.pyplot as plt

model, ckpt, device = load_model(f'{REPO_PATH}/checkpoints/generator.pt')
print(f'Model loaded (epoch={ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.5f})')

_, page = generate_word(model, 'Hello', noise_scale=0.02, device=device)

plt.figure(figsize=(10, 3))
plt.imshow(page, cmap='gray')
plt.axis('off')
plt.title('Generated: Hello')
plt.tight_layout()
plt.show()